In [ ]:
!pip install -q "transformers>=4.46.0" accelerate sentencepiece pandas qwen-vl-utils gdown

# EGCD with canonical split — qwen2_2b

**Model:** `Qwen/Qwen2-VL-2B-Instruct` · **Tag:** `qwen2_2b` · **Platform:** colab

This notebook runs the **full EGCD pipeline** on AMBER's Discriminative-Relation
subset with the **canonical split** (seed=42, stratified by image — identical
across all 7 EGCD notebooks). Replaces the old `qwen2b.ipynb` that ran on the
whole 1,664 set.

For every query in BOTH halves we record:
- `baseline_pred` (greedy Yes/No)
- `cd_pred` (universal CD+APC Yes/No)
- `entropy` (first-token full-vocab Shannon entropy in nats)

τ is selected on the **tuning half** over [0.550, 0.750] step 0.01; the held-out
half is evaluated only at τ* and reported in the paper.

Model loading mirrors `vcd_code/colab/baselines_colab_qwen2_2b_vcd.ipynb` cell 3 exactly so EGCD
numbers are directly comparable to VCD/SID numbers from the same loader.

**Output contract:** `final_entropy_gated_results_qwen2_2b.json` (per-item records
+ tuning sweep + selected τ* + held-out metrics). Drop into `vcd_results/`
alongside the VCD/SID files.


In [ ]:
import os, json, time, re, datetime
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import torch
import torch.nn.functional as F
import numpy as np
from transformers import LogitsProcessor, LogitsProcessorList

ON_KAGGLE = os.path.exists("/kaggle")
BASE_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("/content")

# Colab: also mirror results into Google Drive so a runtime recycle doesn't lose them.
if not ON_KAGGLE:
    RESULTS_DIR = BASE_DIR
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        RESULTS_DIR = Path("/content/drive/MyDrive/EGCD_results")
        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        print("Results will also be saved to Google Drive:", RESULTS_DIR)
    except Exception as e:
        print("Drive mount skipped/failed, results stay in /content:", e)
else:
    RESULTS_DIR = BASE_DIR

AMBER_DIR = BASE_DIR / "AMBER"
IMG_DIR = AMBER_DIR / "image"

if not (AMBER_DIR / "data").exists():
    !git clone https://github.com/junyangwang0410/AMBER.git {AMBER_DIR}
    print("Cloned AMBER repo")
else:
    print("AMBER repo already exists")

if not IMG_DIR.exists() or len(list(IMG_DIR.glob("AMBER_*.jpg"))) < 1000:
    import gdown
    print("Downloading AMBER images from Google Drive...")
    url = "https://drive.google.com/uc?id=1MaCHgtupcZUjf007anNl4_MV0o4DjXvl"
    gdown.download(url, str(BASE_DIR / "AMBER_images.zip"), quiet=False)
    print("Extracting...")
    !unzip -q {BASE_DIR / "AMBER_images.zip"} -d {AMBER_DIR}
    for candidate in [AMBER_DIR, AMBER_DIR / "AMBER", AMBER_DIR / "image"]:
        if candidate.is_dir() and len(list(candidate.glob("AMBER_*.jpg"))) > 100:
            IMG_DIR = candidate
            break

jpgs = list(IMG_DIR.glob("AMBER_*.jpg"))
print(f"Found {len(jpgs)} AMBER images in {IMG_DIR}")
assert len(jpgs) >= 1000, f"Expected >= 1000 images, got {len(jpgs)}"

query_rel = json.load(open(AMBER_DIR / "data/query/query_discriminative-relation.json"))
annotations = json.load(open(AMBER_DIR / "data/annotations.json"))

gt_map, subtype_map = {}, {}
for ann in annotations:
    truth = ann.get("truth")
    if isinstance(truth, str):
        gt_map[ann["id"]] = truth.strip().lower()
        subtype_map[ann["id"]] = ann.get("type")
print(f"Relation queries: {len(query_rel)}")


## Canonical split

Defined once, identical across all 7 EGCD notebooks (seed=42, stratified by
image id so multi-query images never leak between partitions).


In [ ]:
# ============================================================
# CANONICAL SPLIT (identical across all 7 EGCD notebooks).
# seed=42, stratified by IMAGE so multi-query images stay in one
# partition. Yields 831 tuning / 832 held-out query pairs.
# ============================================================
SPLIT_SEED = 42
np.random.seed(SPLIT_SEED)

image_to_qids = {}
for q in query_rel:
    image_to_qids.setdefault(q["image"], []).append(q["id"])
image_names = sorted(image_to_qids.keys())
np.random.shuffle(image_names)
n_images = len(image_names)
cut = n_images // 2
tuning_images = set(image_names[:cut])
held_out_images = set(image_names[cut:])

tuning_ids = set()
held_out_ids = set()
for img, qids in image_to_qids.items():
    target = tuning_ids if img in tuning_images else held_out_ids
    for qid in qids:
        target.add(qid)

tuning_queries = [q for q in query_rel if q["id"] in tuning_ids]
held_out_queries = [q for q in query_rel if q["id"] in held_out_ids]
print(f"Tuning  : {len(tuning_queries)} queries across {len(tuning_images)} images")
print(f"Held-out: {len(held_out_queries)} queries across {len(held_out_images)} images")


## Model loading

Mirrors `vcd_code/colab/baselines_colab_qwen2_2b_vcd.ipynb` cell 3 exactly: same `Qwen/Qwen2-VL-2B-Instruct`, same
processor args, same `(512, 512)` thumbnail.


In [ ]:
# ============================================================
# MODEL LOADING -- qwen2_2b (mirrors vcd_code/ exactly).
# ============================================================
import transformers
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_TAG = "qwen2_2b"
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28
processor = AutoProcessor.from_pretrained(MODEL_NAME, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
model.eval()
TOKENIZER = processor.tokenizer
print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"transformers: {transformers.__version__}")


## Helpers: parse, APC processor, generate functions

Logic copied verbatim from `qwen2b.ipynb` cell 3 (the original EGCD notebook
without split). Only the `image.thumbnail` size and the split-aware loop change.


In [ ]:
PROMPT_SUFFIX = " Answer with only 'Yes' or 'No'."
MAX_NEW_TOKENS = 6
ALPHA = 1.0
BETA = 0.1
SEED = 0
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

NEG_RE = re.compile(r"\b(not|no|none|isn't|aren't|doesn't|don't|wasn't|weren't|isnt|arent|doesnt|dont|wasnt|werent|n't)\b")

def parse_yes_no(response):
    r = response.strip().lower()
    if r.startswith("yes"): return "yes"
    if r.startswith("no"): return "no"
    for word in re.findall(r"[a-z']+", r):
        if word in ("yes", "no"): return word
    if NEG_RE.search(r): return "no"
    return None


class ContrastiveAPCLogitsProcessor(LogitsProcessor):
    """VCD-style APC contrastive processor.

    amateur = counterfactual forward pass with the IMAGE REMOVED
    (text-only conditional). This is the EGCD System 2 amateur,
    identical to the one in the original qwen2b.ipynb.
    """
    def __init__(self, model, cf_prompt_ids, prompt_len_standard, alpha=1.0, beta=0.1):
        self.model = model
        self.cf_prompt_ids = cf_prompt_ids
        self.prompt_len_standard = prompt_len_standard
        self.alpha = alpha
        self.beta = beta

    def __call__(self, input_ids, scores):
        generated_so_far = input_ids[:, self.prompt_len_standard:]
        cf_input_ids = torch.cat([self.cf_prompt_ids, generated_so_far], dim=1)
        with torch.no_grad():
            cf_out = self.model(input_ids=cf_input_ids, use_cache=False)
        cf_logits = cf_out.logits[:, -1, :].float()

        standard_logits = scores.float()
        standard_probs = F.softmax(standard_logits, dim=-1)
        threshold = self.beta * standard_probs.max(dim=-1, keepdim=True).values
        apc_mask = standard_probs >= threshold
        cd_logits = (1 + self.alpha) * standard_logits - self.alpha * cf_logits
        cd_logits = cd_logits.masked_fill(~apc_mask, float("-inf"))
        return cd_logits

try:
    from qwen_vl_utils import process_vision_info
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qwen-vl-utils"], check=True)
    from qwen_vl_utils import process_vision_info

def build_standard_inputs(image, question):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": question + PROMPT_SUFFIX}
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    return processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)

def build_cf_prompt_ids(question):
    messages = [{"role": "user", "content": [
        {"type": "text", "text": question + PROMPT_SUFFIX}
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return TOKENIZER(text, return_tensors="pt").input_ids.to(model.device)


@torch.no_grad()
def get_baseline_with_entropy(image, question, max_new_tokens=MAX_NEW_TOKENS):
    inputs = build_standard_inputs(image, question)
    outputs = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        return_dict_in_generate=True, output_scores=True
    )
    first_token_logits = outputs.scores[0][0]
    probs = F.softmax(first_token_logits.float(), dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-9)).item()

    gen_ids = outputs.sequences[0][inputs["input_ids"].shape[1]:]
    response = processor.decode(gen_ids, skip_special_tokens=True).strip()
    return response, entropy

@torch.no_grad()
def generate_contrastive(image, question, alpha=ALPHA, beta=BETA, max_new_tokens=MAX_NEW_TOKENS):
    standard_inputs = build_standard_inputs(image, question)
    cf_prompt_ids = build_cf_prompt_ids(question)
    prompt_len_standard = standard_inputs["input_ids"].shape[1]

    processor_list = LogitsProcessorList([
        ContrastiveAPCLogitsProcessor(model, cf_prompt_ids, prompt_len_standard, alpha=alpha, beta=beta)
    ])
    out_ids = model.generate(
        **standard_inputs, max_new_tokens=max_new_tokens, do_sample=False,
        logits_processor=processor_list,
    )
    gen = out_ids[0][prompt_len_standard:]
    return processor.decode(gen, skip_special_tokens=True).strip()

def generate_all_signals_entropy(image, question, alpha=ALPHA, beta=BETA):
    baseline_resp, entropy = get_baseline_with_entropy(image, question)
    cd_resp = generate_contrastive(image, question, alpha, beta)
    return parse_yes_no(baseline_resp), parse_yes_no(cd_resp), entropy


In [ ]:
# ============================================================
# SANITY CHECK -- 5 items, no metrics. Verifies the model emits clean
# yes/no and that the entropy value is sensible (nats, > 0).
# ============================================================
print(f"Sanity check for {MODEL_TAG} (5 items)...")
for q in query_rel[:5]:
    img_path = IMG_DIR / q["image"]
    if not img_path.exists():
        print(f"  id={q['id']}: image not found, skipping")
        continue
    image = Image.open(img_path).convert("RGB")
    image.thumbnail(512, 512)
    base_resp, cd_resp, ent = generate_all_signals_entropy(image, q["query"])
    print(f"  id={q['id']} gt={gt_map.get(q['id'])} | base={base_resp} cd={cd_resp} H={ent:.4f}")

print("\nIf all five rows show yes/no and a positive entropy, proceed to the full run.")


## Full run (both halves)

We run all 1,664 queries once. Per-item records are tagged with `split`
(`"tuning"` or `"held_out"`) so the τ sweep and held-out evaluation can be
computed offline from the same file.

Checkpoints every 50 items; safe to stop and re-run this cell.


In [ ]:
RUN_START_T = time.time()
RUN_STARTED_UTC = datetime.datetime.utcnow().isoformat() + "Z"

CHECKPOINT_PATH = RESULTS_DIR / f"egcd_checkpoint_{MODEL_TAG}.json"
CHECKPOINT_LOCAL = BASE_DIR / f"egcd_checkpoint_{MODEL_TAG}.json"
results = []
start_idx = 0
for p in (CHECKPOINT_PATH, CHECKPOINT_LOCAL):
    if p.exists():
        with open(p) as f:
            results = json.load(f)
        start_idx = len(results)
        print(f"Resuming from sample {start_idx} ({p})")
        break

tuning_id_set = set(tuning_ids)
BLANK = {"id": None, "subtype": None, "gt": None, "split": None,
         "baseline_pred": None, "cd_pred": None, "entropy": 0.0,
         "baseline_latency_ms": 0.0, "cd_latency_ms": 0.0}

for i in tqdm(range(start_idx, len(query_rel)), desc=f"EGCD full run ({MODEL_TAG})"):
    q = query_rel[i]
    img_path = IMG_DIR / q["image"]
    gt = gt_map.get(q["id"])
    subtype = subtype_map.get(q["id"])
    if not img_path.exists() or gt is None:
        continue
    image = Image.open(img_path).convert("RGB")
    image.thumbnail(512, 512)

    rec = {"id": q["id"], "subtype": subtype, "gt": gt,
            "split": "tuning" if q["id"] in tuning_id_set else "held_out"}
    try:
        t0 = time.perf_counter()
        base_resp, ent_val = get_baseline_with_entropy(image, q["query"])
        rec["baseline_pred"] = parse_yes_no(base_resp)
        rec["baseline_latency_ms"] = round((time.perf_counter() - t0) * 1000.0, 1)
        rec["entropy"] = float(ent_val)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        rec["baseline_pred"] = None
        rec["entropy"] = 0.0
    try:
        t0 = time.perf_counter()
        cd_resp = generate_contrastive(image, q["query"])
        rec["cd_pred"] = parse_yes_no(cd_resp)
        rec["cd_latency_ms"] = round((time.perf_counter() - t0) * 1000.0, 1)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        rec["cd_pred"] = None
    for k, v in BLANK.items():
        rec.setdefault(k, v)
    results.append(rec)

    if len(results) % 50 == 0:
        for p in (CHECKPOINT_PATH, CHECKPOINT_LOCAL):
            with open(p, "w") as f:
                json.dump(results, f)

for p in (CHECKPOINT_PATH, CHECKPOINT_LOCAL):
    with open(p, "w") as f:
        json.dump(results, f)
print(f"Done. {len(results)} items in {(time.time()-RUN_START_T)/60:.1f} min.")


## τ sweep on tuning half, evaluate on held-out half

This is the actual paper number: τ* is the tuning-half argmax of balanced
accuracy over [0.550, 0.750] step 0.01; held-out metrics are reported only at τ*.


In [ ]:
def calc_metrics(res, threshold):
    """Return (recall, specificity, balanced_acc, n_evaluated, tp, fn, tn, fp) at threshold τ.

    A query is routed to CD iff its entropy >= threshold; otherwise baseline.
    """
    tp = fp = tn = fn = 0
    n_eval = 0
    for r in res:
        pred = r["cd_pred"] if r["entropy"] >= threshold else r["baseline_pred"]
        if pred is None: continue
        gt = r["gt"]
        if r["subtype"] == "discriminative-relation" and gt == "yes":
            n_eval += 1
            if pred == "yes": tp += 1
            else: fn += 1
        elif r["subtype"] == "relation" and gt == "no":
            n_eval += 1
            if pred == "no": tn += 1
            else: fp += 1
    recall = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0.0
    spec  = tn / (tn + fp) * 100 if (tn + fp) > 0 else 0.0
    return recall, spec, (recall + spec) / 2, n_eval, tp, fn, tn, fp

tuning_res = [r for r in results if r["split"] == "tuning"]
held_out_res = [r for r in results if r["split"] == "held_out"]
print(f"Tuning  records: {len(tuning_res)}")
print(f"Held-out records: {len(held_out_res)}")

# Baseline and pure CD+APC on both halves (no threshold gating)
print("\n=== Reference regimes (no gating) ===")
for split_name, res in [("TUNING", tuning_res), ("HELD-OUT", held_out_res)]:
    rec_b, spec_b, bal_b, n_b, *_ = calc_metrics(res, threshold=999.0)
    rec_c, spec_c, bal_c, n_c, *_ = calc_metrics(res, threshold=-1.0)
    print(f"  {split_name:8s} Baseline  R={rec_b:6.2f}  S={spec_b:6.2f}  BalAcc={bal_b:6.2f}  (n={n_b})")
    print(f"  {split_name:8s} CD+APC    R={rec_c:6.2f}  S={spec_c:6.2f}  BalAcc={bal_c:6.2f}  (n={n_c})")

print("\n=== τ sweep on TUNING half (selection) ===")
print(f"{'τ':<8} | {'Recall':<8} | {'Spec':<8} | {'BalAcc':<8} | {'n_eval':<6}")
print("-" * 50)
sweep_data = []
best_tau = None
best_bal = -1.0
for t in np.linspace(0.55, 0.75, 21):
    r, s, b, n, *_ = calc_metrics(tuning_res, float(t))
    sweep_data.append({"threshold": float(t), "recall": r, "specificity": s, "balanced_acc": b, "n_eval": n})
    print(f"{t:<8.3f} | {r:<8.2f} | {s:<8.2f} | {b:<8.2f} | {n:<6}")
    if b > best_bal:
        best_bal = b
        best_tau = float(t)

print(f"\nSelected τ* = {best_tau:.3f}  (tuning-half balanced accuracy = {best_bal:.2f})")

print("\n=== HELD-OUT evaluation at τ* ===")
rec_h, spec_h, bal_h, n_h, tp_h, fn_h, tn_h, fp_h = calc_metrics(held_out_res, best_tau)
rec_b, spec_b, bal_b, n_b, *_ = calc_metrics(held_out_res, 999.0)
rec_c, spec_c, bal_c, n_c, *_ = calc_metrics(held_out_res, -1.0)
print(f"  Held-out Baseline  R={rec_b:6.2f}  S={spec_b:6.2f}  BalAcc={bal_b:6.2f}  (n={n_b})")
print(f"  Held-out CD+APC    R={rec_c:6.2f}  S={spec_c:6.2f}  BalAcc={bal_c:6.2f}  (n={n_c})")
print(f"  Held-out EGCD@τ*   R={rec_h:6.2f}  S={spec_h:6.2f}  BalAcc={bal_h:6.2f}  (n={n_h})  τ*={best_tau:.3f}")
print(f"  Confusion @ τ*: TP={tp_h} FN={fn_h} TN={tn_h} FP={fp_h}")


In [ ]:
# ============================================================
# FINAL SAVE -- final_entropy_gated_results_qwen2_2b.json
# Schema matches the original qwen2b.ipynb output PLUS the split field,
# PLUS latencies, PLUS held-out metrics + confusion counts at τ*.
# ============================================================
meta = {
    "schema": "egcd_split_v1",
    "method": "EGCD (entropy-gated contrastive decoding) with canonical split",
    "model": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "platform": "kaggle" if ON_KAGGLE else "colab",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "dtype": "float16",
    "max_new_tokens": MAX_NEW_TOKENS,
    "alpha": ALPHA,
    "beta": BETA,
    "seed": SEED,
    "split_seed": SPLIT_SEED,
    "split_policy": "stratified_by_image; seed=42; identical across all 7 EGCD notebooks",
    "n_tuning": len(tuning_res),
    "n_held_out": len(held_out_res),
    "image_cap": "thumbnail (512, 512)",
    "prompt_suffix": PROMPT_SUFFIX,
    "amateur": "counterfactual forward pass with image omitted (text-only conditional logits)",
    "selected_tau": best_tau,
    "held_out_metrics": {
        "baseline": {"recall": rec_b, "specificity": spec_b, "balanced_acc": bal_b, "n": n_b},
        "cd_apc":   {"recall": rec_c, "specificity": spec_c, "balanced_acc": bal_c, "n": n_c},
        "egcd":     {"recall": rec_h, "specificity": spec_h, "balanced_acc": bal_h, "n": n_h,
                      "tp": tp_h, "fn": fn_h, "tn": tn_h, "fp": fp_h},
    },
    "tuning_sweep": sweep_data,
    "run_started_utc": RUN_STARTED_UTC,
    "wall_time_min": round((time.time() - RUN_START_T) / 60.0, 1),
}

final = {**meta, "raw_results": results}

FINAL_NAME = f"final_entropy_gated_results_{MODEL_TAG}.json"
for d in (RESULTS_DIR, BASE_DIR):
    with open(Path(d) / FINAL_NAME, "w") as f:
        json.dump(final, f, indent=2)
print("Saved:")
print("  ", Path(RESULTS_DIR) / FINAL_NAME)
print("  ", Path(BASE_DIR) / FINAL_NAME)

if not ON_KAGGLE:
    try:
        from google.colab import files
        files.download(str(Path(BASE_DIR) / FINAL_NAME))
    except Exception as e:
        print("Auto-download skipped:", e)
